# 🛠️ Phase 3: Kỹ thuật Đặc trưng & Data Freeze v1 (TV2 — Nhóm 8, Đề tài 20)
**Nối tiếp trực tiếp từ bản bàn giao `tv1_v1` của TV1 (Thịnh) trong repo [`BAC164/BDA-Nhom_8`](https://github.com/BAC164/BDA-Nhom_8):**  
1. **Môi trường đồng nhất với TV1:** Java 17 + `pyspark==4.0.1`.
2. **Đầu vào:** Đọc và kiểm tra checksum từ `tv1_v1/splits/{train,val,test}` trên Google Drive (`376,749` / `81,606` / `81,028` dòng). Nếu chưa kết nối Drive, notebook tự tái tạo chính xác phép chia hash `SHA256('split:42:' + row_id)` và `SHA256('fold:42:' + row_id)` của Thịnh.
3. **Tự động xuất `src/features.py`:** Chứa đầy đủ pipeline đặc trưng v1, `Length_invalid`, `OOF Target Encoding` (gồm cả `oof_prior` từ 4 folds còn lại), và hai hàm `make_pipeline_lr`, `make_pipeline_gbt(max_bins=300)`.
4. **Kiểm thử chống rò rỉ nhãn & Đóng băng `features_v1`:** Lưu bảng đặc trưng + `lookup_maps/` + `manifest.json` + `split_checksums.json` lên Drive cho TV3, TV4, TV5.

In [ ]:
# Ô 1: Cài đặt Java 17 giống hệt notebook 02_Read_Shared_Data.ipynb của TV1 (Thịnh)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    java_candidates = sorted(Path('/usr/lib/jvm').glob('java-17-openjdk-*'))
    if not java_candidates:
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'openjdk-17-jdk-headless'], check=True)
        java_candidates = sorted(Path('/usr/lib/jvm').glob('java-17-openjdk-*'))
    assert java_candidates, 'Không tìm thấy Java 17!'
    java17_path = java_candidates[0]
    os.environ['JAVA_HOME'] = str(java17_path)
    os.environ['PATH'] = str(java17_path / 'bin') + ':' + os.environ['PATH']
    subprocess.run([str(java17_path / 'bin/java'), '-version'], check=True)
else:
    print('[LOCAL MODE] Đang chạy trên máy cá nhân.')

In [ ]:
# Ô 2: Cài đặt PySpark 4.0.1 (kiểm tra metadata trước khi import để chạy ngay lần đầu không cần Restart)
import importlib
import importlib.metadata

if IN_COLAB:
    try:
        installed_ver = importlib.metadata.version('pyspark')
    except importlib.metadata.PackageNotFoundError:
        installed_ver = None

    if installed_ver != '4.0.1':
        print(f'[SETUP] Phiên bản trên đĩa: {installed_ver} -> Đang cài đặt pyspark==4.0.1 ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==4.0.1'])
        importlib.invalidate_caches()

    # Nếu lỡ nạp phiên bản cũ vào bộ nhớ trước đó mà chưa khởi động Spark JVM, xóa cache module để nạp bản 4.0.1 mới
    if 'pyspark' in sys.modules and getattr(sys.modules['pyspark'], '__version__', None) != '4.0.1':
        for mod_name in list(sys.modules):
            if mod_name == 'pyspark' or mod_name.startswith(('pyspark.', 'py4j')):
                sys.modules.pop(mod_name, None)
        importlib.invalidate_caches()

    os.makedirs('/content/tv2_project/src', exist_ok=True)
    os.makedirs('/content/tv2_project/data', exist_ok=True)
    os.chdir('/content/tv2_project')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
os.makedirs(PROJECT_ROOT / 'src', exist_ok=True)
os.makedirs(PROJECT_ROOT / 'data', exist_ok=True)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType

os.environ['PYSPARK_PYTHON'] = sys.executable
if IN_COLAB:
    assert pyspark.__version__ == '4.0.1', (
        f'Đang nạp PySpark {pyspark.__version__} từ trước trong RAM. Vui lòng chọn Runtime -> Restart session and run all 1 lần.'
    )

spark = (
    SparkSession.builder
    .appName('Airlines_TV2_Features_v1')
    .master('local[2]')
    .config('spark.driver.memory', '4g')
    .config('spark.driver.maxResultSize', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.autoBroadcastJoinThreshold', '20971520')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Phiên bản Spark đang chạy:', spark.version, '| Driver memory:', spark.sparkContext._conf.get('spark.driver.memory', 'default'))

In [ ]:
# Ô 3: Đọc & kiểm tra checksum tv1_v1 từ Drive (hoặc tự tái tạo phép chia SHA256 y hệt TV1 nếu chưa gắn Drive)
import hashlib
import json
import shutil
import urllib.request

# Đổi USE_DRIVE = False nếu muốn chạy độc lập không cần cấp quyền Google Drive
USE_DRIVE = True
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/BigData_Topic20/01_data_frozen/tv1_v1'),
    Path('/content/drive/MyDrive/tv1_v1'),
]

def compute_folder_checksums(folder: Path) -> dict:
    result = {}
    for path in sorted(folder.rglob('*')):
        if path.is_file():
            h = hashlib.sha256()
            with path.open('rb') as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    h.update(block)
            result[path.relative_to(folder).as_posix()] = h.hexdigest()
    return result

BASE = None
DRIVE_MOUNTED = False
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = True
        for cand in DRIVE_CANDIDATES:
            if cand.is_dir() and (cand / 'manifest.json').is_file():
                BASE = cand
                break
    except Exception as e:
        print(f'[WARN] Chưa kết nối Drive ({e}). Chuyển sang chế độ tự tái tạo splits chuẩn tv1_v1.')

SPLITS_LOCAL = PROJECT_ROOT / 'data' / 'splits'

if BASE is not None:
    print(f'[DRIVE] Đã tìm thấy thư mục bàn giao của Thịnh: {BASE}')
    manifest = json.loads((BASE / 'manifest.json').read_text(encoding='utf-8'))
    expected_hashes = json.loads((BASE / 'split_checksums.json').read_text(encoding='utf-8'))
    assert manifest['source']['openml_id'] == 1169
    assert manifest['counts'] == {'train': 376749, 'val': 81606, 'test': 81028}
    assert set(expected_hashes) == {'train', 'val', 'test'}

    # Sao chép splits từ Drive về đĩa cục bộ Colab để Spark đọc nhanh và kiểm tra checksum
    parts = {}
    for name in ('train', 'val', 'test'):
        drive_folder = BASE / 'splits' / name
        local_folder = SPLITS_LOCAL / name
        if not local_folder.exists():
            shutil.copytree(drive_folder, local_folder)
        assert (local_folder / '_SUCCESS').is_file(), f'Thiếu/chưa hoàn tất tập {name}'
        assert compute_folder_checksums(local_folder) == expected_hashes[name], f'{name}: sai checksum so với bản của TV1!'
        part = spark.read.parquet(str(local_folder))
        assert part.schema == StructType.fromJson(manifest['schemas'][name]), f'{name}: sai schema'
        assert part.count() == manifest['counts'][name], f'{name}: sai số dòng'
        parts[name] = part
        print(f'  -> {name}: {manifest["counts"][name]:,} dòng — KHỚP CHECKSUM TV1')

    train_raw, val_raw, test_raw = parts['train'], parts['val'], parts['test']
else:
    print('[FALLBACK] Chưa thấy tv1_v1 trên Drive -> Tự tái tạo chính xác phép chia SHA256 của Thịnh (376,749 / 81,606 / 81,028)...')
    if not (SPLITS_LOCAL / 'train').exists():
        import pandas as pd
        raw_pq = PROJECT_ROOT / 'data' / 'dataset_1169.pq'
        if not raw_pq.exists():
            openml_url = 'https://data.openml.org/datasets/0000/1169/dataset_1169.pq'
            print(f'[DOWNLOAD] Đang tải OpenML 1169 từ {openml_url} ...')
            urllib.request.urlretrieve(openml_url, raw_pq)

        df_pd = pd.read_parquet(raw_pq).reset_index(drop=True)
        df_pd.insert(0, 'row_id', range(len(df_pd)))
        df_pd['Airline'] = df_pd['Airline'].astype(str)
        df_pd['Flight'] = df_pd['Flight'].astype(int).astype(str)
        df_pd['AirportFrom'] = df_pd['AirportFrom'].astype(str)
        df_pd['AirportTo'] = df_pd['AirportTo'].astype(str)
        df_pd['DayOfWeek'] = df_pd['DayOfWeek'].astype(int).astype(str)
        df_pd['Time'] = df_pd['Time'].astype(float)
        df_pd['Length'] = df_pd['Length'].astype(float)
        df_pd['Delay'] = df_pd['Delay'].astype(int)
        staging_pq = PROJECT_ROOT / 'data' / 'staging_with_rowid.pq'
        df_pd.to_parquet(staging_pq, index=False)

        clean_fixed = (
            spark.read.parquet(str(staging_pq))
            .withColumn('Length_invalid', F.when(F.col('Length') <= 0, 1).otherwise(0))
            .withColumn('Length', F.when(F.col('Length_invalid') == 1, F.lit(None).cast('double')).otherwise(F.col('Length')))
        )
        split_hash = F.sha2(F.concat(F.lit('split:42:'), F.col('row_id').cast('string')), 256)
        split_bucket = F.pmod(F.conv(F.substring(split_hash, 1, 8), 16, 10).cast('long'), F.lit(10000))
        assigned = (
            clean_fixed.withColumn('_b', split_bucket)
            .withColumn('_s', F.when(F.col('_b') < 7000, 'train').when(F.col('_b') < 8500, 'val').otherwise('test'))
            .cache()
        )
        tr_base = assigned.filter(F.col('_s') == 'train').drop('_b', '_s')
        va_base = assigned.filter(F.col('_s') == 'val').drop('_b', '_s')
        te_base = assigned.filter(F.col('_s') == 'test').drop('_b', '_s')

        length_median = float(tr_base.approxQuantile('Length', [0.5], 0.0)[0])
        fold_hash = F.sha2(F.concat(F.lit('fold:42:'), F.col('row_id').cast('string')), 256)
        fold_num = F.pmod(F.conv(F.substring(fold_hash, 1, 8), 16, 10).cast('long'), F.lit(5)).cast('int')

        tr_base.fillna({'Length': length_median}).withColumn('fold', fold_num).write.mode('overwrite').parquet(str(SPLITS_LOCAL / 'train'))
        va_base.fillna({'Length': length_median}).write.mode('overwrite').parquet(str(SPLITS_LOCAL / 'val'))
        te_base.fillna({'Length': length_median}).write.mode('overwrite').parquet(str(SPLITS_LOCAL / 'test'))

    train_raw = spark.read.parquet(str(SPLITS_LOCAL / 'train'))
    val_raw   = spark.read.parquet(str(SPLITS_LOCAL / 'val'))
    test_raw  = spark.read.parquet(str(SPLITS_LOCAL / 'test'))
    assert train_raw.count() == 376749 and val_raw.count() == 81606 and test_raw.count() == 81028
    print('  -> Đã tái tạo chính xác số dòng TV1: Train 376,749 | Val 81,606 | Test 81,028')

## Ô 4: Tự động tạo file `src/features.py` chuẩn của TV2
Lệnh `%%writefile src/features.py` ghi trực tiếp toàn bộ module xử lý đặc trưng và 2 hàm `make_pipeline_lr`, `make_pipeline_gbt` ra file `src/features.py` để TV3 và TV4 dùng chung.

In [ ]:
%%writefile src/features.py
"""
src/features.py — Module Kỹ thuật Đặc trưng & Pipeline (Chủ sở hữu: TV2 - Phase 3)
Đề tài BDA_20 (Nhóm 8): Dự báo chuyến bay bị trễ (OpenML ID 1169 - Airlines)

Đầu vào (từ Phase 2 - TV1 / bản bàn giao tv1_v1):
  - splits/{train,val,test} (Parquet)
    Các cột: row_id (long), Airline (string), Flight (string), AirportFrom (string),
             AirportTo (string), DayOfWeek (string), Time (double), Length (double),
             Delay (int: 0/1), Length_invalid (int: 0/1), và cột fold (int: 0..4, chỉ có trong train).

Đầu ra (Phase 3 - Data Freeze v1):
  1. Bảng đặc trưng Parquet: data/features/v1/{train,val,test}/
  2. Bảng tra cứu (Lookup Maps) cho Demo (Phase 6): data/features/v1/lookup_maps/
  3. Hai hàm tạo Pipeline chuẩn cho TV3 & TV4:
     - make_pipeline_lr(...)
     - make_pipeline_gbt(max_bins=300, ...)
  4. Hàm kiểm thử chống rò rỉ nhãn (Leakage Test): verify_no_label_leakage(...)
"""

import os
import json
from typing import Tuple, Dict, Optional, List
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator


# ==============================================================================
# 1. CẤU HÌNH MẶC ĐỊNH PHASE 3 (DATA FREEZE V1)
# ==============================================================================
SEED = 42
NUM_FOLDS = 5
SMOOTHING_M = 30.0           # Hệ số làm mượt m-estimate cho Target Encoding (20..50)
MIN_COUNT_AIRPORT = 100      # Ngưỡng gộp nhóm hiếm OTHER cho sân bay (từ đường cong độ phủ EDA)
MIN_COUNT_ROUTE = 50         # Ngưỡng gộp nhóm hiếm OTHER cho tuyến bay (route)


# ==============================================================================
# 2. CÁC HÀM BIẾN ĐỔI ĐẶC TRƯNG THÀNH PHẦN
# ==============================================================================
FORBIDDEN_PREDICTORS = {"Delay", "row_id", "fold"}


def ensure_schema_and_fold(df: DataFrame, is_train: bool = False, seed: int = SEED) -> DataFrame:
    """
    Đảm bảo kiểu dữ liệu chuẩn khớp bản bàn giao tv1_v1 từ Phase 2:
      - Delay -> int
      - Length_invalid -> int (giữ nguyên cờ 4 dòng Length <= 0 từ TV1; nếu chưa có sẽ tự tạo)
      - DayOfWeek, Flight, Airline, AirportFrom, AirportTo -> string
      - Time, Length -> double
      - row_id -> long
      - Nếu là tập train và chưa có cột 'fold', tự động gán fold theo SHA256('fold:42:' + row_id) % 5.
    """
    out = (
        df
        .withColumn("Airline", F.col("Airline").cast(StringType()))
        .withColumn("Flight", F.col("Flight").cast("int").cast(StringType()))
        .withColumn("AirportFrom", F.col("AirportFrom").cast(StringType()))
        .withColumn("AirportTo", F.col("AirportTo").cast(StringType()))
        .withColumn("DayOfWeek", F.col("DayOfWeek").cast("int").cast(StringType()))
        .withColumn("Time", F.col("Time").cast(DoubleType()))
        .withColumn("Length", F.col("Length").cast(DoubleType()))
        .withColumn("Delay", F.col("Delay").cast(IntegerType()))
    )
    if "Length_invalid" in out.columns:
        out = out.withColumn("Length_invalid", F.col("Length_invalid").cast(IntegerType()))
    else:
        out = out.withColumn(
            "Length_invalid",
            F.when(F.col("Length").isNull() | (F.col("Length") <= 0), F.lit(1)).otherwise(F.lit(0)).cast(IntegerType())
        )
    if "row_id" not in out.columns:
        out = out.withColumn("row_id", F.monotonically_increasing_id())
    else:
        out = out.withColumn("row_id", F.col("row_id").cast("long"))
    if is_train and "fold" not in out.columns:
        fold_hash = F.sha2(F.concat(F.lit(f"fold:{seed}:"), F.col("row_id").cast("string")), 256)
        out = out.withColumn(
            "fold",
            F.pmod(F.conv(F.substring(fold_hash, 1, 8), 16, 10).cast("long"), F.lit(NUM_FOLDS)).cast(IntegerType())
        )
    return out


def add_basic_features(df: DataFrame) -> DataFrame:
    """
    Nhiệm vụ 1: Tạo các đặc trưng cơ bản:
      - hour (double): floor(Time / 60) -> dùng cho GBT
      - hour_cat (string): chuỗi "0".."23" -> dùng cho One-Hot Encoding trong LR
      - time_bucket (string): morning (5-11), afternoon (12-16), evening (17-20), night (21-4)
      - is_weekend (double): 1.0 nếu DayOfWeek thuộc {"6", "7"}, ngược lại 0.0
      - len_bin (string): short (<=90), medium (91-180), long (181-300), ultra_long (>300)
      - route (string): AirportFrom + "_" + AirportTo
      - airline_flight (string): Airline + "_" + Flight
    """
    hour_col = F.floor(F.col("Time") / 60.0)
    return (
        df
        .withColumn("hour", hour_col.cast(DoubleType()))
        .withColumn("hour_cat", hour_col.cast(IntegerType()).cast(StringType()))
        .withColumn(
            "time_bucket",
            F.when((hour_col >= 5) & (hour_col <= 11), F.lit("morning"))
             .when((hour_col >= 12) & (hour_col <= 16), F.lit("afternoon"))
             .when((hour_col >= 17) & (hour_col <= 20), F.lit("evening"))
             .otherwise(F.lit("night"))
        )
        .withColumn(
            "is_weekend",
            F.when(F.col("DayOfWeek").isin("6", "7"), F.lit(1.0)).otherwise(F.lit(0.0)).cast(DoubleType())
        )
        .withColumn(
            "len_bin",
            F.when(F.col("Length") <= 90, F.lit("short"))
             .when(F.col("Length") <= 180, F.lit("medium"))
             .when(F.col("Length") <= 300, F.lit("long"))
             .otherwise(F.lit("ultra_long"))
        )
        .withColumn("route", F.concat_ws("_", F.col("AirportFrom"), F.col("AirportTo")))
        .withColumn("airline_flight", F.concat_ws("_", F.col("Airline"), F.col("Flight")))
    )


def compute_flight_density(
    train_df: DataFrame,
    val_df: DataFrame,
    test_df: DataFrame,
    fit_on_train_only: bool = True
) -> Tuple[DataFrame, DataFrame, DataFrame, DataFrame, DataFrame]:
    """
    Nhiệm vụ 2: Mật độ chuyến bay (Flight Density):
      - density_from: số chuyến bay theo (AirportFrom, DayOfWeek, hour)
      - density_to  : số chuyến bay theo (AirportTo, DayOfWeek, hour)
    Mặc định fit_on_train_only=True để tuân thủ nghiêm ngặt quy tắc của TV1 (02_Read_Shared_Data.ipynb):
    "Fit các bước học từ dữ liệu chỉ trên Train".
    Cắt đứt lineage bằng localCheckpoint + broadcast join để tránh phình LogicalPlan.
    """
    cols_needed = ["AirportFrom", "AirportTo", "DayOfWeek", "hour"]
    if fit_on_train_only:
        schedule_source = train_df.select(cols_needed)
    else:
        schedule_source = (
            train_df.select(cols_needed)
            .unionByName(val_df.select(cols_needed))
            .unionByName(test_df.select(cols_needed))
        )

    dens_from_map = (
        schedule_source
        .groupBy("AirportFrom", "DayOfWeek", "hour")
        .agg(F.count("*").cast(DoubleType()).alias("density_from"))
        .localCheckpoint(eager=True)
    )
    dens_to_map = (
        schedule_source
        .groupBy("AirportTo", "DayOfWeek", "hour")
        .agg(F.count("*").cast(DoubleType()).alias("density_to"))
        .localCheckpoint(eager=True)
    )

    def attach_density(d: DataFrame) -> DataFrame:
        return (
            d
            .join(F.broadcast(dens_from_map), on=["AirportFrom", "DayOfWeek", "hour"], how="left")
            .join(F.broadcast(dens_to_map), on=["AirportTo", "DayOfWeek", "hour"], how="left")
            .fillna({"density_from": 0.0, "density_to": 0.0})
        )

    return attach_density(train_df), attach_density(val_df), attach_density(test_df), dens_from_map, dens_to_map


def fit_and_apply_rare_grouping(
    train_df: DataFrame,
    val_df: DataFrame,
    test_df: DataFrame,
    min_count_airport: int = MIN_COUNT_AIRPORT,
    min_count_route: int = MIN_COUNT_ROUTE
) -> Tuple[DataFrame, DataFrame, DataFrame, Dict[str, List[str]]]:
    """
    Nhiệm vụ 3: Gộp nhóm hiếm (Rare Category Grouping):
      - Các category xuất hiện ít hơn N dòng trong tập TRAIN được gộp thành "OTHER".
      - Tạo các cột: AirportFrom_g, AirportTo_g, route_g.
      - Dùng broadcast join thay cho .isin(2431 strings) để cây Catalyst luôn gọn nhẹ.
    """
    valid_from_df = (
        train_df.groupBy("AirportFrom")
        .count()
        .filter(F.col("count") >= min_count_airport)
        .select("AirportFrom", F.col("AirportFrom").alias("AirportFrom_g"))
        .localCheckpoint(eager=True)
    )
    valid_to_df = (
        train_df.groupBy("AirportTo")
        .count()
        .filter(F.col("count") >= min_count_airport)
        .select("AirportTo", F.col("AirportTo").alias("AirportTo_g"))
        .localCheckpoint(eager=True)
    )
    valid_route_df = (
        train_df.groupBy("route")
        .count()
        .filter(F.col("count") >= min_count_route)
        .select("route", F.col("route").alias("route_g"))
        .localCheckpoint(eager=True)
    )

    valid_from = [r[0] for r in valid_from_df.select("AirportFrom").collect() if r[0] is not None]
    valid_to = [r[0] for r in valid_to_df.select("AirportTo").collect() if r[0] is not None]
    valid_route = [r[0] for r in valid_route_df.select("route").collect() if r[0] is not None]

    def apply_grouping(d: DataFrame) -> DataFrame:
        return (
            d
            .join(F.broadcast(valid_from_df), on="AirportFrom", how="left")
            .join(F.broadcast(valid_to_df), on="AirportTo", how="left")
            .join(F.broadcast(valid_route_df), on="route", how="left")
            .fillna({"AirportFrom_g": "OTHER", "AirportTo_g": "OTHER", "route_g": "OTHER"})
        )

    rare_maps = {
        "min_count_airport": min_count_airport,
        "min_count_route": min_count_route,
        "valid_AirportFrom": valid_from,
        "valid_AirportTo": valid_to,
        "valid_route": valid_route
    }
    return apply_grouping(train_df), apply_grouping(val_df), apply_grouping(test_df), rare_maps


def fit_and_apply_frequency_encoding(
    train_df: DataFrame,
    val_df: DataFrame,
    test_df: DataFrame
) -> Tuple[DataFrame, DataFrame, DataFrame, Dict[str, DataFrame]]:
    """
    Nhiệm vụ 4: Frequency Encoding:
      - Tính log1p(count trong train) = ln(1 + count_train) cho:
        + AirportFrom    -> log_freq_from
        + AirportTo      -> log_freq_to
        + route          -> log_freq_route
        + airline_flight -> log_freq_flight
      - Category chưa từng thấy (unseen) trong val/test nhận giá trị 0.0.
    """
    specs = [
        ("AirportFrom", "log_freq_from"),
        ("AirportTo", "log_freq_to"),
        ("route", "log_freq_route"),
        ("airline_flight", "log_freq_flight"),
    ]
    freq_tables: Dict[str, DataFrame] = {}
    source_train = train_df

    for col_name, out_col in specs:
        freq_df = (
            source_train.groupBy(col_name)
            .agg(F.log1p(F.count("*").cast(DoubleType())).alias(out_col))
            .localCheckpoint(eager=True)
        )
        freq_tables[col_name] = freq_df
        train_df = train_df.join(F.broadcast(freq_df), on=col_name, how="left").fillna({out_col: 0.0})
        val_df = val_df.join(F.broadcast(freq_df), on=col_name, how="left").fillna({out_col: 0.0})
        test_df = test_df.join(F.broadcast(freq_df), on=col_name, how="left").fillna({out_col: 0.0})

    return train_df, val_df, test_df, freq_tables


def fit_and_apply_oof_target_encoding(
    train_df: DataFrame,
    val_df: DataFrame,
    test_df: DataFrame,
    m: float = SMOOTHING_M
) -> Tuple[DataFrame, DataFrame, DataFrame, float, Dict[str, DataFrame]]:
    """
    Nhiệm vụ 5: Target Encoding Out-of-Fold (OOF) với m-estimate smoothing (kể cả OOF prior):
      - Trên tập TRAIN: cả thống kê nhóm lẫn prior toàn cục (oof_prior) của fold f
        đều được tính CHỈ từ 4 folds còn lại.
      - Trên tập VAL và TEST: dùng tổng và prior của toàn bộ tập TRAIN.
      - Sử dụng localCheckpoint(eager=True) + broadcast join cho từng bảng map để cây Catalyst
        giữ độ sâu tuyến tính O(1), không bao giờ bị tràn Heap Java.
    """
    source_train = train_df
    global_agg = source_train.agg(
        F.sum(F.col("Delay").cast(DoubleType())).alias("total_S"),
        F.count("*").cast(DoubleType()).alias("total_N")
    ).first()
    total_S = float(global_agg["total_S"])
    total_N = float(global_agg["total_N"])
    prior = total_S / total_N

    # Tính oof_prior riêng cho từng fold từ 4 folds còn lại
    fold_totals = (
        source_train.groupBy("fold")
        .agg(
            F.sum(F.col("Delay").cast(DoubleType())).alias("fold_S"),
            F.count("*").cast(DoubleType()).alias("fold_N")
        )
    )
    oof_priors = (
        fold_totals
        .withColumn(
            "oof_prior",
            ((F.lit(total_S) - F.col("fold_S")) / (F.lit(total_N) - F.col("fold_N"))).cast(DoubleType())
        )
        .select("fold", "oof_prior")
        .localCheckpoint(eager=True)
    )

    te_specs = [
        ("route", "te_route"),
        ("airline_flight", "te_flight"),
        ("AirportFrom_g", "te_from"),
        ("AirportTo_g", "te_to"),
    ]
    full_te_maps: Dict[str, DataFrame] = {}

    for col_name, te_col in te_specs:
        # Thống kê theo (category, fold) trên train gốc (không dùng train_df đang bị join dần)
        g = (
            source_train.groupBy(col_name, "fold")
            .agg(
                F.sum(F.col("Delay").cast(DoubleType())).alias("sum_fold"),
                F.count("*").cast(DoubleType()).alias("cnt_fold")
            )
        )
        tot = (
            g.groupBy(col_name)
            .agg(
                F.sum("sum_fold").alias("sum_total"),
                F.sum("cnt_fold").alias("cnt_total")
            )
        )
        oof = (
            g.join(tot, on=col_name, how="inner")
            .join(F.broadcast(oof_priors), on="fold", how="inner")
            .withColumn(
                te_col,
                (
                    (F.col("sum_total") - F.col("sum_fold") + F.lit(m) * F.col("oof_prior"))
                    / (F.col("cnt_total") - F.col("cnt_fold") + F.lit(m))
                ).cast(DoubleType())
            )
            .select(col_name, "fold", te_col)
            .localCheckpoint(eager=True)
        )
        train_df = (
            train_df
            .join(F.broadcast(oof), on=[col_name, "fold"], how="left")
            .join(F.broadcast(oof_priors), on="fold", how="left")
            .withColumn(te_col, F.coalesce(F.col(te_col), F.col("oof_prior")))
            .drop("oof_prior")
        )

        full_map = (
            tot.withColumn(
                te_col,
                ((F.col("sum_total") + F.lit(m * prior)) / (F.col("cnt_total") + F.lit(m))).cast(DoubleType())
            )
            .select(col_name, te_col)
            .localCheckpoint(eager=True)
        )
        full_te_maps[col_name] = full_map
        val_df = val_df.join(F.broadcast(full_map), on=col_name, how="left").fillna({te_col: prior})
        test_df = test_df.join(F.broadcast(full_map), on=col_name, how="left").fillna({te_col: prior})

    return train_df, val_df, test_df, prior, full_te_maps


# ==============================================================================
# 3. HÀM TỔNG HỢP XÂY DỰNG BẢNG ĐẶC TRƯNG V1 & LƯU BẢNG TRA CỨU
# ==============================================================================
FEATURE_TABLE_COLUMNS = [
    # Định danh & Nhãn (không đưa vào features)
    "row_id", "Delay",
    # Biến gốc đã chuẩn hóa (bao gồm cờ Length_invalid từ TV1)
    "Airline", "Flight", "AirportFrom", "AirportTo", "DayOfWeek", "Time", "Length", "Length_invalid",
    # Đặc trưng cơ bản
    "hour", "hour_cat", "time_bucket", "is_weekend", "len_bin", "route", "airline_flight",
    # Mật độ chuyến bay (tính từ tập Train)
    "density_from", "density_to",
    # Nhóm gộp hiếm (OTHER)
    "AirportFrom_g", "AirportTo_g", "route_g",
    # Frequency Encoding
    "log_freq_from", "log_freq_to", "log_freq_route", "log_freq_flight",
    # Out-of-Fold Target Encoding
    "te_route", "te_flight", "te_from", "te_to"
]


def build_features_v1(
    train_raw: DataFrame,
    val_raw: DataFrame,
    test_raw: DataFrame,
    smoothing_m: float = SMOOTHING_M,
    min_count_airport: int = MIN_COUNT_AIRPORT,
    min_count_route: int = MIN_COUNT_ROUTE,
    fit_density_on_train_only: bool = True
) -> Tuple[DataFrame, DataFrame, DataFrame, dict]:
    """
    Thực thi toàn bộ quy trình Kỹ thuật Đặc trưng Phase 3 trên 3 tập train/val/test.
    Trả về: (train_fe, val_fe, test_fe, artifacts_dict)
    """
    train_df = ensure_schema_and_fold(train_raw, is_train=True)
    val_df = ensure_schema_and_fold(val_raw, is_train=False)
    test_df = ensure_schema_and_fold(test_raw, is_train=False)

    # Bước 1: Đặc trưng cơ bản
    train_df = add_basic_features(train_df)
    val_df = add_basic_features(val_df)
    test_df = add_basic_features(test_df)

    # Bước 2: Gộp nhóm hiếm thành OTHER (chỉ tính ngưỡng trên train) và cắt lineage để tái sử dụng nhanh
    train_df, val_df, test_df, rare_maps = fit_and_apply_rare_grouping(
        train_df, val_df, test_df,
        min_count_airport=min_count_airport,
        min_count_route=min_count_route
    )
    train_df = train_df.localCheckpoint(eager=True)

    # Bước 3: Mật độ chuyến bay (mặc định fit trên tập train)
    train_df, val_df, test_df, dens_from_map, dens_to_map = compute_flight_density(
        train_df, val_df, test_df, fit_on_train_only=fit_density_on_train_only
    )

    # Bước 4: Frequency Encoding (log1p count trên train)
    train_df, val_df, test_df, freq_tables = fit_and_apply_frequency_encoding(train_df, val_df, test_df)

    # Bước 5: Out-of-Fold Target Encoding với smoothing m
    train_df, val_df, test_df, prior, full_te_maps = fit_and_apply_oof_target_encoding(
        train_df, val_df, test_df, m=smoothing_m
    )

    # Sắp xếp cột và đóng gói bảng đã tính toán xong bằng localCheckpoint để cắt toàn bộ LogicalPlan cũ
    train_cols = FEATURE_TABLE_COLUMNS + ["fold"]
    train_out = train_df.select(*train_cols).localCheckpoint(eager=True)
    val_out = val_df.select(*FEATURE_TABLE_COLUMNS).localCheckpoint(eager=True)
    test_out = test_df.select(*FEATURE_TABLE_COLUMNS).localCheckpoint(eager=True)

    artifacts = {
        "prior": prior,
        "smoothing_m": smoothing_m,
        "rare_maps": rare_maps,
        "freq_tables": freq_tables,
        "full_te_maps": full_te_maps,
        "dens_from_map": dens_from_map,
        "dens_to_map": dens_to_map,
    }
    return train_out, val_out, test_out, artifacts


def save_features_and_lookups(
    train_fe: DataFrame,
    val_fe: DataFrame,
    test_fe: DataFrame,
    artifacts: dict,
    output_base_dir: str = "data/features/v1"
) -> None:
    """
    Ghi 3 bảng đặc trưng ra Parquet (KHÔNG ghi cột đã qua StringIndexer để giữ nguyên tính tổng quát)
    và lưu toàn bộ bảng tra cứu (lookup_maps) phục vụ Demo ở Phase 6.
    """
    os.makedirs(output_base_dir, exist_ok=True)
    for split_name, df_part in [("train", train_fe), ("val", val_fe), ("test", test_fe)]:
        out_path = os.path.join(output_base_dir, split_name)
        df_part.write.mode("overwrite").parquet(out_path)
        print(f"[SAVE] Đã ghi tập {split_name} ra: {out_path}")

    lookup_dir = os.path.join(output_base_dir, "lookup_maps")
    os.makedirs(lookup_dir, exist_ok=True)

    # Lưu metadata & danh sách nhóm hợp lệ (JSON)
    meta_payload = {
        "prior": artifacts["prior"],
        "smoothing_m": artifacts["smoothing_m"],
        "min_count_airport": artifacts["rare_maps"]["min_count_airport"],
        "min_count_route": artifacts["rare_maps"]["min_count_route"],
        "valid_AirportFrom": artifacts["rare_maps"]["valid_AirportFrom"],
        "valid_AirportTo": artifacts["rare_maps"]["valid_AirportTo"],
        "valid_route": artifacts["rare_maps"]["valid_route"],
    }
    with open(os.path.join(lookup_dir, "encoding_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta_payload, f, indent=2, ensure_ascii=False)

    # Lưu các bảng tra cứu Target Encoding, Frequency Encoding, Flight Density
    for col_name, df_map in artifacts["full_te_maps"].items():
        df_map.write.mode("overwrite").parquet(os.path.join(lookup_dir, f"te_{col_name}.parquet"))

    for col_name, df_map in artifacts["freq_tables"].items():
        df_map.write.mode("overwrite").parquet(os.path.join(lookup_dir, f"freq_{col_name}.parquet"))

    artifacts["dens_from_map"].write.mode("overwrite").parquet(os.path.join(lookup_dir, "density_from.parquet"))
    artifacts["dens_to_map"].write.mode("overwrite").parquet(os.path.join(lookup_dir, "density_to.parquet"))
    print(f"[SAVE] Đã lưu đầy đủ bảng tra cứu (Lookup Maps) tại: {lookup_dir}")


# ==============================================================================
# 4. HAI HÀM TẠO PIPELINE CHUẨN CHO TV3 (LR) VÀ TV4 (GBT)
# ==============================================================================
def make_pipeline_lr(
    reg_param: float = 0.0,
    elastic_net_param: float = 0.0,
    max_iter: int = 100,
    include_te: bool = False,
    include_length_invalid: bool = True,
    include_estimator: bool = True
) -> Pipeline:
    """
    Nhiệm vụ 6A: Tạo Pipeline chuẩn cho TV3 (Logistic Regression).
    - Biến phân loại đưa vào OneHotEncoder: Airline, DayOfWeek, hour_cat (giờ ở dạng chuỗi).
    - Biến số: Length, log_freq_from, log_freq_to, log_freq_route, density_from, density_to
      (+ Length_invalid nếu include_length_invalid=True).
    - Tuyệt đối không đưa Delay, row_id, fold vào danh sách cột đầu vào của VectorAssembler.
    """
    cat_cols = ["Airline", "DayOfWeek", "hour_cat"]
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_i", handleInvalid="keep")
        for c in cat_cols
    ]
    ohe = OneHotEncoder(
        inputCols=[f"{c}_i" for c in cat_cols],
        outputCols=[f"{c}_oh" for c in cat_cols]
    )

    num_cols = [
        "Length",
        "log_freq_from",
        "log_freq_to",
        "log_freq_route",
        "density_from",
        "density_to"
    ]
    if include_length_invalid:
        num_cols = num_cols + ["Length_invalid"]
    if include_te:
        num_cols = num_cols + ["is_weekend", "te_route", "te_flight", "te_from", "te_to"]

    leaked = FORBIDDEN_PREDICTORS.intersection(cat_cols + num_cols)
    assert not leaked, f"Cấm đưa cột {leaked} vào predictors của LR!"

    assembler = VectorAssembler(
        inputCols=[f"{c}_oh" for c in cat_cols] + num_cols,
        outputCol="features"
    )

    stages = indexers + [ohe, assembler]
    if include_estimator:
        lr = LogisticRegression(
            labelCol="Delay",
            featuresCol="features",
            maxIter=max_iter,
            regParam=reg_param,
            elasticNetParam=elastic_net_param,
            standardization=True
        )
        stages.append(lr)

    return Pipeline(stages=stages)


def make_pipeline_gbt(
    max_bins: int = 300,
    max_iter: int = 100,
    max_depth: int = 5,
    step_size: float = 0.1,
    subsampling_rate: float = 1.0,
    seed: int = SEED,
    include_length_invalid: bool = True,
    include_estimator: bool = True
) -> Pipeline:
    """
    Nhiệm vụ 6B: Tạo Pipeline chuẩn cho TV4 (GBTClassifier).
    - Biến phân loại: Airline, DayOfWeek, AirportFrom_g, AirportTo_g.
      StringIndexer nằm trực tiếp trong cùng Pipeline với VectorAssembler & GBTClassifier
      để giữ nguyên metadata 'nominal' cho các nút cây quyết định.
    - max_bins mặc định = 300 (>= số lượng category tối đa của AirportFrom_g/AirportTo_g).
    - Tuyệt đối không đưa Delay, row_id, fold vào danh sách cột đầu vào của VectorAssembler.
    """
    cat_cols = ["Airline", "DayOfWeek", "AirportFrom_g", "AirportTo_g"]
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_i", handleInvalid="keep")
        for c in cat_cols
    ]

    num_cols = [
        "hour",
        "Length",
        "te_route",
        "te_flight",
        "log_freq_route",
        "density_from",
        "density_to"
    ]
    if include_length_invalid:
        num_cols = num_cols + ["Length_invalid"]

    leaked = FORBIDDEN_PREDICTORS.intersection(cat_cols + num_cols)
    assert not leaked, f"Cấm đưa cột {leaked} vào predictors của GBT!"

    assembler = VectorAssembler(
        inputCols=[f"{c}_i" for c in cat_cols] + num_cols,
        outputCol="features"
    )

    stages = indexers + [assembler]
    if include_estimator:
        gbt = GBTClassifier(
            labelCol="Delay",
            featuresCol="features",
            maxIter=max_iter,
            maxDepth=max_depth,
            stepSize=step_size,
            subsamplingRate=subsampling_rate,
            maxBins=max_bins,
            seed=seed
        )
        stages.append(gbt)

    return Pipeline(stages=stages)


# ==============================================================================
# 5. KIỂM THỬ CHỐNG RÒ RỈ NHÃN (NHIỆM VỤ 8 - BẮT BUỘC)
# ==============================================================================
def verify_no_label_leakage(
    train_raw: DataFrame,
    train_fe: DataFrame,
    val_fe: DataFrame,
    prior: float,
    m: float = SMOOTHING_M
) -> dict:
    """
    Nhiệm vụ 8: Thực hiện 3 bài kiểm thử bắt buộc để chứng minh KHÔNG rò rỉ nhãn (Label Leakage):
      1. Đối chiếu thủ công giá trị te_route trên tập VAL với công thức (S + m*prior)/(N + m) tính trực tiếp từ train_raw.
      2. Đối chiếu thủ công giá trị OOF te_route trên tập TRAIN với công thức dùng 4 folds còn lại (gồm cả OOF prior).
      3. Huấn luyện mô hình chỉ gồm duy nhất đặc trưng `te_route` rồi so sánh ROC-AUC trên Train và ROC-AUC trên Val.
    """
    print("\n=== BẮT ĐẦU KIỂM THỬ CHỐNG RÒ RỈ NHÃN (LABEL LEAKAGE CHECKS) ===")

    # --- Kiểm thử 1: Đối chiếu thủ công 5 tuyến bay trong tập VAL ---
    sample_routes = [
        r["route"] for r in val_fe.select("route", "te_route").distinct().limit(5).collect()
    ]
    train_with_route = ensure_schema_and_fold(train_raw, is_train=True).withColumn(
        "route", F.concat_ws("_", F.col("AirportFrom"), F.col("AirportTo"))
    )
    manual_checks = []
    max_abs_diff = 0.0

    for rt in sample_routes:
        stat = (
            train_with_route.filter(F.col("route") == rt)
            .agg(
                F.sum(F.col("Delay").cast(DoubleType())).alias("S"),
                F.count("*").cast(DoubleType()).alias("N")
            )
            .first()
        )
        S_val = float(stat["S"]) if stat["S"] is not None else 0.0
        N_val = float(stat["N"]) if stat["N"] is not None else 0.0
        expected_te = (S_val + m * prior) / (N_val + m) if N_val > 0 else prior

        actual_te = float(
            val_fe.filter(F.col("route") == rt).select("te_route").first()["te_route"]
        )
        diff = abs(expected_te - actual_te)
        max_abs_diff = max(max_abs_diff, diff)
        manual_checks.append({
            "route": rt,
            "train_S": S_val,
            "train_N": N_val,
            "manual_te": round(expected_te, 6),
            "pipeline_val_te": round(actual_te, 6),
            "abs_diff": diff
        })
        print(f"  [Check 1 - Val] Route={rt:<10} | Manual={expected_te:.6f} | Val_TE={actual_te:.6f} | Diff={diff:.2e}")

    assert max_abs_diff < 1e-6, f"PHÁT HIỆN SAI LỆCH TE_ROUTE TRÊN VAL: max_abs_diff = {max_abs_diff}"

    # --- Kiểm thử 2: Đối chiếu thủ công OOF te_route (gồm OOF prior từ 4 folds còn lại) trên tập TRAIN ---
    oof_samples = train_fe.select("route", "fold", "te_route").distinct().limit(3).collect()
    max_oof_diff = 0.0
    for row in oof_samples:
        rt, fld, actual_oof = row["route"], int(row["fold"]), float(row["te_route"])
        other_folds = train_with_route.filter(F.col("fold") != fld)
        other_global = other_folds.agg(
            F.avg(F.col("Delay").cast(DoubleType())).alias("oof_prior")
        ).first()
        oof_prior = float(other_global["oof_prior"])
        other_rt = other_folds.filter(F.col("route") == rt).agg(
            F.sum(F.col("Delay").cast(DoubleType())).alias("S_oof"),
            F.count("*").cast(DoubleType()).alias("N_oof")
        ).first()
        s_oof = float(other_rt["S_oof"]) if other_rt["S_oof"] is not None else 0.0
        n_oof = float(other_rt["N_oof"]) if other_rt["N_oof"] is not None else 0.0
        expected_oof = (s_oof + m * oof_prior) / (n_oof + m)
        diff_oof = abs(expected_oof - actual_oof)
        max_oof_diff = max(max_oof_diff, diff_oof)
        print(f"  [Check 2 - OOF] Route={rt:<10} fold={fld} | Manual_OOF={expected_oof:.6f} | Train_TE={actual_oof:.6f} | Diff={diff_oof:.2e}")

    assert max_oof_diff < 1e-6, f"PHÁT HIỆN SAI LỆCH OOF TE_ROUTE TRÊN TRAIN: max_oof_diff = {max_oof_diff}"

    # --- Kiểm thử 3: Kiểm tra mô hình đơn biến chỉ gồm `te_route` (So sánh AUC Train vs Val) ---
    asm_single = VectorAssembler(inputCols=["te_route"], outputCol="features_single")
    lr_single = LogisticRegression(
        labelCol="Delay", featuresCol="features_single", maxIter=20
    )
    single_model = Pipeline(stages=[asm_single, lr_single]).fit(train_fe)

    evaluator = BinaryClassificationEvaluator(
        labelCol="Delay", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    )
    auc_train_single = float(evaluator.evaluate(single_model.transform(train_fe)))
    auc_val_single = float(evaluator.evaluate(single_model.transform(val_fe)))
    auc_gap = auc_train_single - auc_val_single

    print(f"  [Check 3 - AUC] Single-Feature (te_route) ROC-AUC -> Train: {auc_train_single:.4f} | Val: {auc_val_single:.4f} | Gap: {auc_gap:+.4f}")

    leakage_passed = (max_abs_diff < 1e-6) and (max_oof_diff < 1e-6) and (abs(auc_gap) < 0.03)
    status_str = "PASSED (Không phát hiện rò rỉ nhãn)" if leakage_passed else "WARNING (Cần kiểm tra lại)"
    print(f"=== KẾT LUẬN KIỂM THỬ RÒ RỈ NHÃN: {status_str} ===\n")

    return {
        "status": status_str,
        "max_manual_abs_diff": max_abs_diff,
        "max_oof_abs_diff": max_oof_diff,
        "single_feature_te_route_auc_train": round(auc_train_single, 4),
        "single_feature_te_route_auc_val": round(auc_val_single, 4),
        "auc_gap_train_minus_val": round(auc_gap, 4),
        "manual_route_checks": manual_checks
    }

## Ô 5: Xây dựng Bảng Đặc trưng v1 (`build_features_v1`)
- Giữ nguyên `Length_invalid`, `row_id`, `Delay` (không đưa `Delay`, `row_id`, `fold` vào predictors).
- Tính mật độ chuyến bay (`density_from`, `density_to`) và tần suất (`log_freq_*`), ngưỡng gộp nhóm `OTHER` (`AirportFrom_g`, `AirportTo_g`, `route_g`) **chỉ từ tập Train**.
- Tính **Out-of-Fold Target Encoding (`te_route`, `te_flight`, `te_from`, `te_to`) với cả `oof_prior` từ 4 folds còn lại**.

In [ ]:
import importlib
import features
importlib.reload(features)

from features import (
    build_features_v1,
    save_features_and_lookups,
    verify_no_label_leakage,
    make_pipeline_lr,
    make_pipeline_gbt,
    FORBIDDEN_PREDICTORS
)

train_fe, val_fe, test_fe, artifacts = build_features_v1(
    train_raw, val_raw, test_raw,
    smoothing_m=30.0,
    min_count_airport=100,
    min_count_route=50,
    fit_density_on_train_only=True
)

train_fe.cache()
val_fe.cache()
print('Schema bảng đặc trưng v1:')
train_fe.printSchema()
train_fe.select(
    'row_id', 'Airline', 'route', 'Length', 'Length_invalid',
    'hour', 'hour_cat', 'density_from', 'AirportFrom_g',
    'log_freq_route', 'te_route', 'te_flight', 'Delay'
).show(5, truncate=False)

## Ô 6: Kiểm thử bắt buộc — Chống rò rỉ nhãn (`verify_no_label_leakage`)
1. **Check 1 (Val):** Đối chiếu thủ công `te_route` trên tập `val` với công thức `(S + m*prior)/(N + m)` từ `train_raw`.
2. **Check 2 (OOF Train):** Đối chiếu thủ công `te_route` trên tập `train` với công thức Out-of-Fold gồm cả `oof_prior` tính từ 4 folds còn lại.
3. **Check 3 (AUC):** Huấn luyện mô hình đơn biến chỉ gồm `te_route` và so sánh `ROC-AUC` trên `train` với `val`.

In [ ]:
leakage_report = verify_no_label_leakage(
    train_raw=train_raw,
    train_fe=train_fe,
    val_fe=val_fe,
    prior=artifacts['prior'],
    m=artifacts['smoothing_m']
)
assert 'PASSED' in leakage_report['status'], 'Kiểm thử chống rò rỉ nhãn chưa đạt!'

## Ô 7: Lưu Parquet `features_v1` + `lookup_maps/` + Checksum và chạy Smoke Test
Ghi dữ liệu v1 ra đĩa cục bộ, tạo `manifest.json` & `split_checksums.json`, rồi đọc lại từ Parquet để kiểm thử `make_pipeline_lr().fit()` và `make_pipeline_gbt(max_bins=300).fit()`.

In [ ]:
FEATURES_V1_DIR = PROJECT_ROOT / 'data' / 'features' / 'v1'
save_features_and_lookups(train_fe, val_fe, test_fe, artifacts, output_base_dir=str(FEATURES_V1_DIR))

# Tạo manifest và checksums chuẩn như TV1 để TV3, TV4, TV5 kiểm tra khi đọc lại
v1_parts = {
    'train': spark.read.parquet(str(FEATURES_V1_DIR / 'train')),
    'val':   spark.read.parquet(str(FEATURES_V1_DIR / 'val')),
    'test':  spark.read.parquet(str(FEATURES_V1_DIR / 'test')),
}
v1_counts = {k: df.count() for k, df in v1_parts.items()}
assert v1_counts == {'train': 376749, 'val': 81606, 'test': 81028}, f'Sai số dòng: {v1_counts}'

v1_checksums = {k: compute_folder_checksums(FEATURES_V1_DIR / k) for k in ('train', 'val', 'test')}
v1_manifest = {
    'version': 'features_v1',
    'tag': 'data-freeze-v1',
    'parent_data': 'tv1_v1',
    'counts': v1_counts,
    'schemas': {k: df.schema.jsonValue() for k, df in v1_parts.items()},
    'prior_train_full': artifacts['prior'],
    'smoothing_m': artifacts['smoothing_m'],
    'min_count_airport': artifacts['rare_maps']['min_count_airport'],
    'min_count_route': artifacts['rare_maps']['min_count_route'],
    'leakage_check': leakage_report['status'],
}
(FEATURES_V1_DIR / 'manifest.json').write_text(json.dumps(v1_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(FEATURES_V1_DIR / 'split_checksums.json').write_text(json.dumps(v1_checksums, indent=2), encoding='utf-8')

# Smoke Test đúng như cách TV3 và TV4 sẽ chạy ở Phase 4
tr_sample = v1_parts['train'].sample(False, 0.05, seed=42)
val_sample = v1_parts['val'].sample(False, 0.05, seed=42)

print('[SMOKE TEST TV3] Đang chạy make_pipeline_lr().fit(tr_sample)...')
m_lr = make_pipeline_lr(max_iter=10).fit(tr_sample)
p_lr = m_lr.transform(val_sample)
print('  -> TV3 Pipeline LR ĐẠT! Số dòng dự đoán mẫu:', p_lr.count())

print('[SMOKE TEST TV4] Đang chạy make_pipeline_gbt(max_bins=300, max_iter=5).fit(tr_sample)...')
m_gbt = make_pipeline_gbt(max_bins=300, max_iter=5, max_depth=4).fit(tr_sample)
p_gbt = m_gbt.transform(val_sample)
print('  -> TV4 Pipeline GBT ĐẠT! Số dòng dự đoán mẫu:', p_gbt.count())

## Ô 8: Đồng bộ `features_v1` lên Google Drive của nhóm & đối chiếu checksum

In [ ]:
import time

if IN_COLAB and DRIVE_MOUNTED:
    # Ưu tiên lưu vào BigData_Topic20/01_data_frozen/features_v1 nếu tồn tại; nếu chưa tạo thư mục gốc thì lưu vào MyDrive/features_v1
    parent_frozen = Path('/content/drive/MyDrive/BigData_Topic20/01_data_frozen')
    if parent_frozen.is_dir():
        drive_out = parent_frozen / 'features_v1'
    else:
        drive_out = Path('/content/drive/MyDrive/features_v1')
    try:
        # Xóa bản cũ trên Drive (nếu có) để tránh trộn file Parquet có tên UUID khác nhau từ các lần chạy trước
        if drive_out.exists():
            shutil.rmtree(drive_out)
        shutil.copytree(FEATURES_V1_DIR, drive_out)
        time.sleep(3)
        for split_name in ('train', 'val', 'test'):
            assert compute_folder_checksums(drive_out / split_name) == v1_checksums[split_name], f'Lỗi checksum trên Drive ở tập {split_name}!'
        print(f'[DRIVE SUCCESS] Đã đồng bộ sạch sẽ và xác minh checksum features_v1 tại: {drive_out}')
    except Exception as exc:
        print(f'[DRIVE WARN] Không thể ghi/đối chiếu tại {drive_out} ({exc}). Bạn có thể tải từ {FEATURES_V1_DIR}.')
else:
    print(f'[LOCAL READY] Dữ liệu đóng băng v1 sẵn sàng tại: {FEATURES_V1_DIR.resolve()}')

print('[HOÀN TẤT PHASE 3 — TV2]')
print('  - File cần push lên GitHub (nhánh tv2/features): src/features.py, feature_spec.md, notebooks/03_features_tv2.ipynb')
# Bỏ dấu # ở dòng dưới nếu muốn tải trực tiếp src/features.py từ Colab về máy:
# if IN_COLAB:
#     from google.colab import files; files.download(str(PROJECT_ROOT / 'src' / 'features.py'))